<a href="https://colab.research.google.com/github/DelCuyos/FUNDAI-Lab7/blob/main/Lab7_Intro_to_ML_Cuyos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    adjusted_rand_score,
  )

In [2]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

df = pd.read_csv(GITHUB_RAW_URL)

print("Dataset shape:", df.shape)
print()
print(df.head())
print()
print("Class distribution:")
print(df ["species"].value_counts())

Dataset shape: (150, 5)

   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

Class distribution:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [4]:
  feature_columns = [
      "sepal length",
      "sepal_width",
      "petal_length",
      "petal_width"
  ]

  X = df[feature_columns].values
  y = df["species"].values

  X_train, X_test, y_train, y_test = train_test_split(
      X,
      y,
      test_size=0.3,
      stratify=y,
      random_state=42
  )

  print("Training samples:", X_train.shape)
  print("Testing samples:", X_test.shape)

KeyError: "['sepal length'] not in index"

## Part A: Supervised Learning (Labels ARE Used)
The model learns a mapping from features to known labels.
Scaling is placed INSIDE the pipeline so that each cross-validation fold computes scaling statistics from its training portion only.
This prevents data leakage.

In [ ]:
supervised_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42))
])

supervised_pipeline.fit(X_train, y_train)

supervised_predictions = supervised_pipeline.predict(X_test)

print("SUPERVISED LEARNING RESULTS")
print("=" * 50)
print("Test accuracy:", accuracy_score(y_test, supervised_predictions))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, supervised_predictions))
print()
print(classification_report(y_test, supervised_predictions))

## Part B: Unsupervised Learning (Labels ARE NOT Used)
The species column is completely ignored.
K-Means must discover group structure on its own.
Afterwards, we compare the discovered clusters to the true labels only for evaluation purposes.

In [ ]:
#NOTE: y is NOT used anywhere in this part.
X_unlabeled = df[feature_columns].values

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_unlabeled)

comparison = pd.crosstab(
    pd.Series(cluster_labels, name="Discovered Cluster"),
    pd.Series(y, name="True Species")
)

print("UNSUPERVISED LEARNING RESULTS")
print("=" * 50)
print("Cluster vs. True Species crosstab:")
print(comparison)
print()
print("Adjusted Rand Index (cluster agreement with true labels):")
print(adjusted_rand_score(y, cluster_labels))

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_unlabeled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="viridis", edgecolor="k")
plt.title("K-Means Clusters (unsupervised)")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.subplot(1, 2, 2)
species_codes = pd.factorize(y)[0]
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=species_codes, cmap="viridis", edgecolor="k")
plt.title("True Species (for comparison only)")
plt.xlabel("PC1")

plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

## Part C: Complexity Sweep (Bias-Variance Demonstration)
We train decision trees with increasing max_depth.

- Very small depth -> high bias -> underfitting
- Moderate depth -> sweet spot -> good generalization
- Very large depth -> high variance
-> overfitting (train = 100%, gap widens)

In [ ]:
depth_values = list(range(1, 16))

train_scores = []
test_scores = []

for depth in depth_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=depth, random_state=42))
    ])

    pipe.fit(X_train, y_train)

    train_scores.append(accuracy_score(y_train, pipe.predict(X_train)))
    test_scores.append(accuracy_score(y_test, pipe.predict(X_test)))

sweep_table = pd.DataFrame({
    "max_depth": depth_values,
    "train_accuracy": train_scores,
    "test_accuracy": test_scores,
    "gap": np.array(train_scores) - np.array(test_scores)
})

print(sweep_table.to_string(index=False))

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(depth_values, train_scores, marker="o", label="Training accuracy")
plt.plot(depth_values, test_scores, marker="s", label="Testing accuracy")

plt.axvspan(1, 2, alpha=0.15, color="red", label="Underfitting region")
plt.axvspan(3, 6, alpha=0.15, color="green", label="Sweet spot")
plt.axvspan(7, 15, alpha=0.15, color="orange", label="Overfitting region")

plt.xlabel("Tree max_depth (model complexity)")
plt.ylabel("Accuracy")
plt.title("Bias-Variance Tradeoff: Train vs. Test Accuracy")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Observed Regions (fill from your sweep table)
Underfitting depth(s): [e.g., 1-2], because train and test accuracy are both low.
Sweet-spot depth(s): (e.g., 3-61, because test accuracy is highest and the gap is small.
Overfitting depth(s): [e.g., 8-15], because train accuracy reaches 1.0 while the train test gap widens.

##Part D: 5-Fold Stratified Cross-Validation
Cross-validation is run ONLY on the training set.
The held-out test set is never used for model selection.
Mean standard deviation shows both performance and stability.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
  "Shallow tree (depth=1, high bias)": Pipeline([
      ("scaler", StandardScaler()),
      ("model", DecisionTreeClassifier(max_depth=1, random_state z = 42 )
  ]),
  "Moderate tree (depth=4)": Pipeline([
      ("scaler", StandardScaler),
      ("model", DecisionTreeClassifier(max_depth=4, random_state=42))
  ]),
  "Full tree (no depth limit, high variance)": Pipeline(
  ("scaler", StandardScaler(1),
  ("model", DecisionTreeClassifier(max_depth=None, random_state e = 42 )
  ]),
}

cv_results =
for name, model in models.items():
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
cv_results.append({
"Model": name,
"CV mean": scores.mean),
"CV sta": scores.std(),
"Fold scores": np.round(scores, 3)
}}
print(name)
print(" Fold scores:", np.round(scores, 3))
print(" Mean std (scores.mean():.3f) (scores.std():.3f)")
print(""60)
cv_table = pd.DataFrame(cv_results)
print(cv_table.to_string(index=False))

### Data Leakage Demonstration (DO NOT use in real projects)
Below, scaling is fitted on ALL data (including the test set) BEFORE cross-validation. This leaks test information into training and usually produces optimistically biased scores. Compare with the pipeline results above.

In [ ]:
leaky_scaler StandardScaler().fit(X)
X_leaky leaky_scaler.transform(X)
#WRONG: uses test data too
Leaky_model DecisionTreeClassifier(max_depth=4, random_state=42)
leaky_scores = cross_val_score (leaky_model, X_leaky, y, cv=cv, scoring="accuracy")
print("Leaky CV mean:", leaky_scores.mean())
print("Difference vs. honest pipeline is usually small on this easy dataset,")
print("but on harder datasets leakage inflates scores dangerously.")